In [37]:
import math
import torch
import random
import numpy as np
from torch import nn, optim
import json

from collections import defaultdict
from datetime import datetime
from dataclasses import dataclass, asdict
from tokenizer import TinyStoriesTokenizer

from torch.utils.data import Sampler, SubsetRandomSampler
from torch.utils.data import Dataset, DataLoader
from tokenizer import TinyStoriesTokenizer

# ============= Hyper-parameters for training ============== #

@dataclass
class Config :
    vocab_size: int = 5000  # This number should agree with the tokenizer
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 1
    vector_dim: int = 256
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1


class TinyStoriesDataset(Dataset):
    def __init__(self, data_file, block_size):
        """
        data_file: path to the .bin file (uint16 array of token IDs)
        block_size: the context window (e.g., 256 or 512 tokens)
        """

        # Memory-map the data file (RAM usage stays near zero!)
        self.data = np.memmap(data_file, dtype=np.uint16, mode='r')
        self.block_size = block_size

    def __len__(self):
        # We subtract block_size to ensure we don't go out of bounds
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Pull a chunk of length block_size + 1 (data and target)
        chunk = self.data[idx : idx + self.block_size + 1]

        # Convert to torch tensors
        x = torch.from_numpy(chunk[:-1].astype(np.int64)) # Input
        y = torch.from_numpy(chunk[1:].astype(np.int64))  # Target (shifted by 1)

        return x, y 




In [38]:
!pip install spacy
#!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_lg
import spacy


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 184.3 MB/s  0:00:0200:0100:01

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [39]:
def align_tags_with_bpe(tokens_bpe, tokens_spacy, tags_spacy):
    bpe_tags_aligned = []
    idx_spacy = 0  # recorrer word spacy

    acumulado_bpe = ""
    acumulado_spacy = ""
    tag_actual = None
    
    for token in tokens_bpe:
        token_limpio = token.replace(" ", "")
        acumulado_bpe += token_limpio
        
        if not token_limpio:
            bpe_tags_aligned.append("X")
            continue

        is_first_subtoken = False
            
        # If our spacy separates token in 2 dif. Then bpe_acum will be ahead spacy
        while len(acumulado_bpe) > len(acumulado_spacy) and idx_spacy < len(tokens_spacy):
            acumulado_spacy += tokens_spacy[idx_spacy].replace(" ", "")
            tag_actual = tags_spacy[idx_spacy]
            idx_spacy += 1
            # if in -> this tokes is the begining or whole
            is_first_subtoken = True

        if is_first_subtoken:
            bpe_tags_aligned.append(tag_actual)
        else:
            bpe_tags_aligned.append("SUBWORD")
        
    return bpe_tags_aligned

def tensor_tags(ids_tokens_dataset, tokenizer, nlp, tag2id):
    tokens_bpe = tokenizer.decode_to_tokens(ids_tokens_dataset)
    text = tokenizer.decode(ids_tokens_dataset)
    
    # SpaCy
    doc = nlp(text)
    tokens_spacy = [t.text for t in doc]
    tags_spacy = [t.pos_ for t in doc]
    tags_aligned = align_tags_with_bpe(tokens_bpe, tokens_spacy, tags_spacy)
    
    # strings tags to id
    # if no tag - None - 'X'
    tags_ids = []
    for tag in tags_aligned:
        if tag in tag2id:
            tags_ids.append(tag2id[tag])
        else:
            tags_ids.append(tag2id['X'])

    return torch.tensor(tags_ids, dtype=torch.long)

In [40]:

#### PROCESAMIENTO TAGS
import os
import torch
import spacy
from tqdm import tqdm

POS_TAGS_VOCAB = [
    'PAD', 'SUBWORD', 'X', 'ADJ', 'ADP', 'ADV', 'AUX', 'CONJ', 
    'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 
    'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB'
]
tag2id = {tag: idx for idx, tag in enumerate(POS_TAGS_VOCAB)}
id2tag = {idx: tag for tag, idx in tag2id.items()}

#nlp = spacy.load("en_core_web_sm")
nlp = spacy.load("en_core_web_lg")

### IMPO
# Spacy divides day!John" internally
infixes = nlp.Defaults.infixes + [r'(?<=[0-9a-zA-Z])([\.!\?,])(?=[0-9a-zA-Z])']
infix_re = spacy.util.compile_infix_regex(infixes)
nlp.tokenizer.infix_finditer = infix_re.finditer
####

tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', Config.block_size)

# Output configurations
FINAL_FILE = 'tags_probes_5k_first_subtoken.pt'
NUM_PROBE_STORIES = 5000 #
dataset_tag_tensors = []

# 3. Resume progress from checkpoint if available
if os.path.exists(FINAL_FILE):
    print(f"Existing file detected. Loading previous progress...")
    dataset_tag_tensors = torch.load(FINAL_FILE)
    processed_stories = len(dataset_tag_tensors)
    print(f"Already have {processed_stories} stories ready.")
else:
    processed_stories = 0
    print("Starting preprocessing from scratch...")

# 4. Main preprocessing loop
if processed_stories < NUM_PROBE_STORIES:
    print(f"Processing blocks from {processed_stories} to {NUM_PROBE_STORIES}:")
    
    # Loop from last checkpoint 
    for idx in tqdm(range(processed_stories, NUM_PROBE_STORIES)):
        try:
            # extratc a window of text of block_size =512. Can be the middle, end,etc
            # idx=0 -> dataset_index = 0      (tokens 0 a 512)
            # idx=1 -> dataset_index = 512    (tokens 512 a 1024)
            # idx=2 -> dataset_index = 1024   (tokens 1024 a 1536)
            dataset_index = idx * Config.block_size
            
            # Validamos no salirnos del tamaño total del dataset de tu profesor
            if dataset_index + Config.block_size >= len(training_dataset):
                print("\n We reached the final of dataset before 5,000 samples")
                break
                
            # call get_item in correct position
            example = training_dataset[dataset_index]
            native_ids_list = example[0].tolist()
            
            # Alignment function
            tag_tensor = tensor_tags(native_ids_list, tokenizer, nlp, tag2id)
            dataset_tag_tensors.append(tag_tensor)
            
            # Checkpoint 
            if (idx + 1) % 500 == 0:
                torch.save(dataset_tag_tensors, FINAL_FILE)
                
        except Exception as e:
            print(f"\n Error processing story index {idx}: {e}")
            print("Saving current progress before quiting...")
            torch.save(dataset_tag_tensors, FINAL_FILE)
            raise e

    torch.save(dataset_tag_tensors, FINAL_FILE)
    print(f"saved with {NUM_PROBE_STORIES} aligned tags.")
else:
    print("\nAll 5000 stories were already fully preprocessed.")

Existing file detected. Loading previous progress...
Already have 5000 stories ready.

All 5000 stories were already fully preprocessed.


In [49]:
### Check aligment
tensores_tags_dataset = torch.load('tags_probes_5k_first_subtoken.pt')

idx = 1 
dataset_index = idx * Config.block_size
example = training_dataset[dataset_index]

ids = example[0].tolist()
tokens = tokenizer.decode_to_tokens(ids)
tags = tensores_tags_dataset[idx]

for i, tid in enumerate(tags.tolist()):    
    tok_txt = tokens[i].replace('\n', '\\n')
    tag_name = id2tag[tid]
    print(f"Story: '{idx}'|Token: '{dataset_index}'|Token_id: '{i}' | Token: '{tok_txt}' | tag: {tid} ({tag_name})")

Story: '1'|Token: '512'|Token_id: '0' | Token: ' at' | tag: 4 (ADP)
Story: '1'|Token: '512'|Token_id: '1' | Token: ' it' | tag: 14 (PRON)
Story: '1'|Token: '512'|Token_id: '2' | Token: '.' | tag: 16 (PUNCT)
Story: '1'|Token: '512'|Token_id: '3' | Token: ' The' | tag: 9 (DET)
Story: '1'|Token: '512'|Token_id: '4' | Token: ' car' | tag: 11 (NOUN)
Story: '1'|Token: '512'|Token_id: '5' | Token: ' felt' | tag: 19 (VERB)
Story: '1'|Token: '512'|Token_id: '6' | Token: ' sad' | tag: 3 (ADJ)
Story: '1'|Token: '512'|Token_id: '7' | Token: ' and' | tag: 8 (CCONJ)
Story: '1'|Token: '512'|Token_id: '8' | Token: ' went' | tag: 19 (VERB)
Story: '1'|Token: '512'|Token_id: '9' | Token: ' away' | tag: 5 (ADV)
Story: '1'|Token: '512'|Token_id: '10' | Token: ' to' | tag: 13 (PART)
Story: '1'|Token: '512'|Token_id: '11' | Token: ' hide' | tag: 19 (VERB)
Story: '1'|Token: '512'|Token_id: '12' | Token: '.' | tag: 16 (PUNCT)
Story: '1'|Token: '512'|Token_id: '13' | Token: ' The' | tag: 9 (DET)
Story: '1'|Toke

In [42]:
#### Verifications

In [43]:
import random 

POS_TAGS_VOCAB = [
    'PAD', 'SUBWORD', 'X', 'ADJ', 'ADP', 'ADV', 'AUX', 'CONJ', 
    'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 
    'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB'
]
tag2id = {tag: idx for idx, tag in enumerate(POS_TAGS_VOCAB)}
id2tag = {idx: tag for tag, idx in tag2id.items()}
tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', Config.block_size)

print(f"Dataset len: {len(training_dataset)} stories")

tensores_tags_dataset = torch.load('tags_probes_5k_first_subtoken.pt')


Dataset len: 18507842 stories


In [44]:
# Check tags to see correct mapping
todos_los_ids_presentes = set()

for i in range(10):
    ids_unicos_historia = tensores_tags_dataset[i].tolist()
    todos_los_ids_presentes.update(ids_unicos_historia)

print("id num in tensors (all ids found):")
print(todos_los_ids_presentes)

print("\n mapping tag2id")
print(tag2id)


id num in tensors (all ids found):
{1, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19}

 mapping tag2id
{'PAD': 0, 'SUBWORD': 1, 'X': 2, 'ADJ': 3, 'ADP': 4, 'ADV': 5, 'AUX': 6, 'CONJ': 7, 'CCONJ': 8, 'DET': 9, 'INTJ': 10, 'NOUN': 11, 'NUM': 12, 'PART': 13, 'PRON': 14, 'PROPN': 15, 'PUNCT': 16, 'SCONJ': 17, 'SYM': 18, 'VERB': 19}


## Create dictionary multiple POS

In [45]:
set_pos = defaultdict(set)
count = 0
for idx in range(len(tensores_tags_dataset)):  
    # real position idx
    dataset_index = idx * Config.block_size
    # extract the corresponding example
    example = training_dataset[dataset_index]
    ids = example[0].tolist()

    # Decode
    tags = tensores_tags_dataset[idx]
    tokens = tokenizer.decode_to_tokens(ids)
    
    for i, tid in enumerate(tags.tolist()):
        tok_txt = tokens[i].replace('\n', '\\n')
        tag_name = id2tag[tid]
        if tid <= 2:
            pass
        else:
            set_pos[tok_txt].add(tid)

# dictionary: [word : [POS_TAG_ID_1, POS_TAG_ID_2,...]
dict_multipos =  {}
for word_set, tag_set in set_pos.items():
    if len(tag_set) > 1:
        dict_multipos[word_set] = list(tag_set)

#print(dict_multipos)

In [47]:
with open("multipos_dict.json", "w", encoding="utf-8") as f:
    json.dump(dict_multipos, f)

with open('multipos_dict.json', 'r') as f:
    dict_multipos = json.load(f)

In [48]:
POS_TAGS_VOCAB = [
    'PAD', 'SUBWORD', 'X', 'ADJ', 'ADP', 'ADV', 'AUX', 'CONJ', 
    'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 
    'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB'
]

tensores_tags_dataset = torch.load('tags_probes_5k_first_subtoken.pt')
tag2id = {tag: idx for idx, tag in enumerate(POS_TAGS_VOCAB)}
id2tag = {idx: tag for tag, idx in tag2id.items()}
tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', Config.block_size)

# Usaremos un defaultdict de sets para evitar duplicados
word_pos_dict = defaultdict(set)

print("Building word dictionaries and POS tags...")
# Usar solo un subconjunto o todo el dataset de los tensores ya procesados
for idx in range(len(tensores_tags_dataset)):
    dataset_index = idx * Config.block_size
    example = training_dataset[dataset_index]
    ids = example[0].tolist()
    
    tags = tensores_tags_dataset[idx].tolist()
    tokens = tokenizer.decode_to_tokens(ids)

    for tok, tag_id in zip(tokens, tags):
        # Ignorar 0 (PAD), 1 (SUBWORD), 2 (X) - Solo queremos clases reales
        if tag_id >= 3:
            # Limpiamos el token de espacios o saltos de linea
            clean_tok = tok.replace(' ', '').replace('\n', '').strip()
            
            # Solo guardamos si el token no quedó vacío
            if clean_tok: 
                if clean_tok.isalpha():
                    tag_name = id2tag[tag_id]
                    word_pos_dict[clean_tok].add(tag_name)

# Convertir sets a listas para que sea compatible con JSON
word_pos_dict_serializable = {word: list(tags) for word, tags in word_pos_dict.items()}

# Guardamos en un archivo
with open('word_to_pos_dict.json', 'w', encoding='utf-8') as f:
    json.dump(word_pos_dict_serializable, f, indent=4)
    print('save pending')
print("Diccionario guardado exitosamente en 'word_to_pos_dict.json'")

# Opcional: Mostrar algunas palabras que tienen múltiples tags (ambiguas)
ambiguous_words = {w: tags for w, tags in word_pos_dict_serializable.items() if len(tags) > 1}
print(f"Total de palabras ambiguas encontradas: {len(ambiguous_words)}")
print("Ambiguous words:", list(ambiguous_words.items())[:50])

Building word dictionaries and POS tags...
save pending
Diccionario guardado exitosamente en 'word_to_pos_dict.json'
Total de palabras ambiguas encontradas: 1604
Ambiguous words: [('day', ['PROPN', 'NOUN']), ('was', ['AUX', 'VERB', 'NOUN']), ('playing', ['VERB', 'NOUN']), ('with', ['ADP', 'SCONJ']), ('his', ['PRON', 'VERB', 'NOUN']), ('red', ['ADJ', 'PROPN', 'NOUN']), ('car', ['VERB', 'NOUN']), ('He', ['ADJ', 'PROPN', 'PRON', 'VERB', 'NOUN']), ('loved', ['AUX', 'VERB', 'ADJ']), ('to', ['VERB', 'PART', 'NOUN', 'ADP', 'AUX']), ('zoom', ['VERB', 'NOUN']), ('it', ['PRON', 'VERB', 'NOUN', 'PROPN']), ('around', ['ADP', 'ADV']), ('the', ['PRON', 'DET']), ('and', ['ADV', 'CCONJ', 'NOUN']), ('make', ['VERB', 'NOUN']), ('But', ['PROPN', 'CCONJ', 'NOUN']), ('this', ['PRON', 'DET']), ('missing', ['ADJ', 'VERB']), ('had', ['AUX', 'VERB']), ('left', ['ADV', 'ADJ', 'VERB', 'NOUN']), ('inside', ['ADV', 'ADJ', 'ADP', 'NOUN']), ('but', ['SCONJ', 'CCONJ']), ('in', ['ADV', 'ADJ', 'SCONJ', 'VERB', 'NOUN', 